# 🇫🇷 Snack4Pets — French F5-TTS UGC Voice Generation

Using **F5-TTS** with the official French community checkpoint from **Issue #434** ([`RASPIAUDIO/F5-French-MixedSpeakers-reduced`](https://huggingface.co/RASPIAUDIO/F5-French-MixedSpeakers-reduced)).

- **Zero English accent bleeding:** Trained directly on 120k French samples with a dedicated French character vocabulary.
- **High-energy UGC cadence:** Flow-Matching Diffusion Transformer replicates breath pauses and rapid conversational tempo.
- **Your video reference:** Automatically clones your creator reference audio (`reference_voices/tao_chew_sample.wav`).

## 1. Setup GPU & Install F5-TTS

In [ ]:
!nvidia-smi
!apt-get -y update && apt-get install -y ffmpeg
%pip install --quiet git+https://github.com/SWivid/F5-TTS.git
%pip install --quiet huggingface_hub soundfile torchaudio

## 2. Clone Snack4Pets Repo (Reference Voice & Audio Clips)

In [ ]:
import os
if not os.path.exists('/content/snack4pets-ugc-tts'):
    !git clone https://github.com/mohaidoss/snack4pets-ugc-tts.git /content/snack4pets-ugc-tts
%cd /content/snack4pets-ugc-tts
!git pull

REF_AUDIO = '/content/snack4pets-ugc-tts/reference_voices/tao_chew_sample.wav'
REF_TEXT = "Tao a mis presque 16 minutes pour la finir. C'est une mastication qui dure entre 10 et 20 minutes en fonction des chiens. Elle fait partie de la catégorie moyenne durée."
print(f"Reference audio loaded: {REF_AUDIO}")

## 3. Download French F5-TTS Weights (Issue #434 Checkpoint)

In [ ]:
from huggingface_hub import hf_hub_download

print("Downloading French F5-TTS model and vocab from RASPIAUDIO...")
CKPT_PATH = hf_hub_download(
    repo_id="RASPIAUDIO/F5-French-MixedSpeakers-reduced",
    filename="model_last_reduced.pt"
)
VOCAB_PATH = hf_hub_download(
    repo_id="RASPIAUDIO/F5-French-MixedSpeakers-reduced",
    filename="vocab.txt"
)
print(f"Model downloaded: {CKPT_PATH}")
print(f"Vocab downloaded: {VOCAB_PATH}")

## 4. Load F5-TTS with French Weights

In [ ]:
from f5_tts.api import F5TTS
from IPython.display import Audio, display
import soundfile as sf

f5_fr = F5TTS(
    model="F5TTS_Base",
    ckpt_file=CKPT_PATH,
    vocab_file=VOCAB_PATH,
    ode_method="euler"
)
print("French F5-TTS pipeline initialized successfully!")

## 5. Generate French UGC Audio for Snack4Pets

- **`speed=1.12`**: Pushes the cadence into genuine TikTok / Reels conversational ad rhythm.
- Punctuation marks (`,`, `.`, `!`, `?`) dictate natural breath timing.

In [ ]:
ad_script = (
    "Si ton chien passe ses journées à s'ennuyer ou à ronger tes meubles, écoute bien ! "
    "Chez Snack4Pets, les friandises sont 100% naturelles, sans aucun additif bizarre. "
    "Ça l'occupe pendant des heures, et ça nettoie ses dents en même temps. "
    "Franchement, le pack mastication a sauvé mon canapé. Teste, tu verras direct la différence !"
)

output_wav = "/content/snack4pets_french_f5.wav"

wav, sr, _ = f5_fr.infer(
    ref_file=REF_AUDIO,
    ref_text=REF_TEXT,
    gen_text=ad_script,
    speed=1.12,
    nfe_step=32
)

sf.write(output_wav, wav, sr)
print(f"Audio generated: {output_wav} ({sr}Hz)")
display(Audio(output_wav))

## 6. Custom Prompt Playground
Try any French sentence below with Tao's voice or your own reference clip.

In [ ]:
custom_text = "Tu t'es déjà demandé ce qui rend nos oreilles de veau si irrésistibles pour les chiens ? C'est simple, c'est du 100% naturel sans aucun produit chimique !"

wav, sr, _ = f5_fr.infer(
    ref_file=REF_AUDIO,
    ref_text=REF_TEXT,
    gen_text=custom_text,
    speed=1.10
)
display(Audio(wav, rate=sr))